In [1]:
# =========================
# 0. IMPORT THƯ VIỆN
# =========================

import os
import re
import unicodedata
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# Tải tài nguyên NLTK cần dùng
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
df = pd.read_csv(r'D:\User\Tài liệu học\HK2 - 2025-2026\1. Data\4. Translator\processed_data_translated_en.csv')
df['Comment_en'] = df['Comment_en'].astype(str).fillna('')
# Loại bỏ các dòng mà Username hoặc nội dung bị lỗi #NAME?
df = df[df['Username'] != '#NAME?']
df = df[df['Comment_en'] != '#NAME?']

In [3]:
# --- 1. CHUẨN BỊ STOPWORDS ---
# Giữ nguyên phần đọc file ở trên.
# Cell này chỉ phục vụ preprocessing.

import unicodedata
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

# Stopwords tiếng Anh mặc định từ NLTK
english_stopwords = set(stopwords.words("english"))

# Custom stopwords bản nhẹ: chỉ bỏ các từ rất chung/rác.
# Nếu sau khi chạy LDA vẫn thấy từ nào nhiễu thì mới thêm tiếp vào đây.
custom_stopwords = {
    # Từ review quá chung
    "place", "places",
    "thing", "things", "something",
    "time", "times",
    "day", "days",
    "hour", "hours",
    "minute", "minutes",
    "lot", "lots",
    "bit",

    # Động từ kể chuyện quá chung
    "go", "going", "went", "gone",
    "come", "came",
    "get", "got", "getting",
    "take", "took", "taken", "taking",
    "make", "made", "making",
    "look", "looking", "looked",
    "see", "seen", "seeing",
    "want", "wanted",
    "think", "thought",
    "know", "knew",

    # Filler / nhấn mạnh
    "really", "quite", "very", "much", "many",
    "maybe", "overall",

    # Không gian/thời gian quá chung
    "near", "nearby",
    "away", "around",
    "inside", "outside",

    # Từ lỗi / nhiễu thấy trong dữ liệu
    "vive", "snus", "ubder", "sinilar", "sobthe",
    "dont", "didnt", "doesnt", "wasnt", "werent", "isnt", "arent"
}

# Nhóm này để tham khảo, CHƯA dùng ngay để tránh lọc quá mạnh.
optional_stopwords = {
    "visit", "visited", "visiting",
    "experience", "experiences",
    "good", "great", "nice", "beautiful", "amazing",
    "wonderful", "lovely", "excellent", "interesting",
    "best", "better",
    "recommend", "recommended", "highly",
    "tour", "tours", "trip", "trips",
    "city", "area", "site", "attraction", "attractions"
}

all_stopwords = english_stopwords.union(custom_stopwords)

print("Số stopwords tiếng Anh mặc định:", len(english_stopwords))
print("Số custom stopwords đang dùng:", len(custom_stopwords))
print("Số optional stopwords chưa dùng:", len(optional_stopwords))
print("Tổng stopwords đang dùng:", len(all_stopwords))

Số stopwords tiếng Anh mặc định: 198
Số custom stopwords đang dùng: 69
Số optional stopwords chưa dùng: 28
Tổng stopwords đang dùng: 266


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [4]:
# --- 2. BIGRAM / TRIGRAM / PHRASE THỦ CÔNG ---
# Cell này KHÔNG xóa từ.
# Nó chỉ ghép các cụm quan trọng thành 1 token:
# ví dụ: old town -> old_town, ticket price -> ticket_price

def remove_accents(text):
    """Bỏ dấu để xử lý các địa danh như 'Tam Coc Bích Dong' -> 'tam coc bich dong'."""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKD", text)
    text = "".join([c for c in text if not unicodedata.combining(c)])
    return text

# Các cụm chắc chắn nên giữ vì có ý nghĩa rõ trong review du lịch.
manual_core_phrases = [
    # Công trình / địa điểm phổ biến
    "dragon bridge",
    "golden bridge",
    "japanese bridge",
    "old quarter",
    "old town",
    "ancient town",
    "marble mountains",
    "cham towers",
    "independence palace",
    "reunification palace",
    "central post office",
    "notre dame cathedral",
    "war remnants museum",
    "temple of literature",
    "hoa lo prison",
    "cu chi tunnels",
    "ha long bay",
    "ba na hills",
    "ben thanh market",
    "hoi an ancient town",
    "saigon central post office",
    "ho chi minh mausoleum",
    "my son sanctuary",
    "my son ruins",
    "my son temple",

    # Cảnh quan / thiên nhiên
    "white sand",
    "clear water",
    "sand dunes",
    "white sand dunes",
    "red sand dunes",
    "sunset view",
    "mountain view",
    "sea view",
    "fairy stream",

    # Trải nghiệm / dịch vụ / chi phí
    "night market",
    "floating market",
    "street food",
    "local food",
    "food court",
    "entrance fee",
    "ticket price",
    "cable car",
    "boat ride",
    "boat trip",
    "boat tour",
    "basket boat",
    "basket boat ride",
    "tour guide",
    "audio guide",
    "bike rental",
    "motorbike rental",
    "souvenir shop",
    "gift shop",
    "photo spot",
    "walking street",
    "theme park",
    "water park",
    "amusement park"
]


# Tự lấy cụm địa danh từ cột Address để không phải liệt kê hết địa danh thủ công.
address_phrases = []
# Mặc định: dùng core phrases + address phrases.
manual_phrases = manual_core_phrases + address_phrases
# Bỏ trùng và xếp cụm dài trước:
# "saigon central post office" phải được thay trước "central post office".
manual_phrases = sorted(
    set([
        remove_accents(p).lower().strip()
        for p in manual_phrases
        if isinstance(p, str) and p.strip()
    ]),
    key=lambda x: len(x.split()),
    reverse=True
)

print("Số manual core phrases:", len(manual_core_phrases))
print("Số address phrases tự lấy từ Address:", len(address_phrases))
print("Tổng manual phrases đang dùng:", len(manual_phrases))

print("\nMột số phrases đang dùng:")
display(pd.DataFrame({"phrase": manual_phrases[:80]}))

Số manual core phrases: 58
Số address phrases tự lấy từ Address: 0
Tổng manual phrases đang dùng: 58

Một số phrases đang dùng:


,phrase
0,saigon central post office
1,hoi an ancient town
2,ho chi minh mausoleum
3,my son sanctuary
4,temple of literature
5,white sand dunes
6,war remnants museum
7,ben thanh market
8,notre dame cathedral
9,my son temple


In [5]:
# --- 3. HÀM GHÉP CỤM + TOKENIZE ---

def normalize_for_phrases(text):
    """
    Chuẩn hóa text trước khi ghép cụm:
    - lowercase
    - bỏ dấu tiếng Việt nếu có
    - bỏ 's
    - đổi dấu gạch ngang / dấu slash thành khoảng trắng
    - giữ chữ, số, khoảng trắng và dấu _
    """
    if not isinstance(text, str):
        return ""

    text = remove_accents(text)
    text = text.lower()

    # Ho Chi Minh's -> ho chi minh
    text = re.sub(r"['’]s\b", "", text)

    # Tam Coc-Bich Dong -> Tam Coc Bich Dong
    text = re.sub(r"[-/]", " ", text)

    # Bỏ ký tự đặc biệt, giữ dấu _
    text = re.sub(r"[^a-z0-9_\s]", " ", text)

    # Gộp khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text


def apply_manual_phrases(text, address=None):
    """
    Ghép cụm thủ công:
    old town -> old_town
    ticket price -> ticket_price
    cable car -> cable_car

    Bản này KHÔNG tự lấy Address nữa để tránh ghép cụm rác.
    """

    text = normalize_for_phrases(text)

    # Chỉ dùng các cụm mình đã chủ động khai báo
    phrases = list(manual_phrases)

    # Chuẩn hóa, bỏ trùng, xếp cụm dài trước
    phrases = [normalize_for_phrases(p) for p in phrases if isinstance(p, str)]
    phrases = sorted(set(phrases), key=lambda x: len(x.split()), reverse=True)

    for phrase in phrases:
        if not phrase:
            continue

        phrase_token = phrase.replace(" ", "_")

        pattern = r"\b" + r"\s+".join(map(re.escape, phrase.split())) + r"\b"

        text = re.sub(pattern, phrase_token, text)

    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize_before_stopwords(text, address=None):
    """Dùng để kiểm tra token trước khi xóa stopwords."""
    text = apply_manual_phrases(text, address)
    return text.split()


def clean_and_tokenize(text, address=None):
    """
    Pipeline cuối:
    1. Ghép cụm thủ công
    2. Tách token
    3. Xóa dấu "_" thừa ở đầu/cuối token
    4. Lemmatize token thường
    5. Xóa stopwords
    6. Giữ token đủ dài
    """
    text = apply_manual_phrases(text, address)
    tokens = text.split()

    cleaned_tokens = []

    for token in tokens:
        # Xóa dấu "_" thừa ở đầu/cuối
        # Ví dụ: "_cheap_" -> "cheap", "boat_" -> "boat"
        token = token.strip("_")

        # Nếu sau khi strip mà rỗng thì bỏ
        if not token:
            continue

        # Nếu là phrase có dấu "_" ở giữa, giữ nguyên
        # Ví dụ: "cable_car", "entrance_fee"
        if "_" in token:
            final_token = token
        else:
            final_token = lemmatizer.lemmatize(token)

        # Xóa stopword
        if final_token in all_stopwords:
            continue

        # Xóa token quá ngắn
        if len(final_token.replace("_", "")) <= 2:
            continue

        cleaned_tokens.append(final_token)

    return cleaned_tokens


In [6]:
# --- 4. ÁP DỤNG PREPROCESSING ---
# Không đọc lại file ở đây. Cell này dùng luôn df đã được đọc ở phần đầu notebook.

COMMENT_COL = "Comment_en"
ADDRESS_COL = "Address"

# Kiểm tra cột cần dùng
missing_cols = [col for col in [COMMENT_COL, ADDRESS_COL] if col not in df.columns]
if missing_cols:
    raise ValueError(f"Thiếu cột: {missing_cols}. Hãy kiểm tra lại tên cột trong file CSV.")

# Token trước khi xóa stopwords, để kiểm tra có bị lọc quá mạnh không
df["tokens_before_stopwords"] = df.apply(
    lambda row: tokenize_before_stopwords(row[COMMENT_COL], row[ADDRESS_COL]),
    axis=1
)

# Text sau khi ghép phrase, dùng để kiểm tra trung gian
df["processed_text_raw_phrase"] = df.apply(
    lambda row: apply_manual_phrases(row[COMMENT_COL], row[ADDRESS_COL]),
    axis=1
)

# Token cuối cùng dùng cho LDA
df["processed_text_Final"] = df.apply(
    lambda row: clean_and_tokenize(row[COMMENT_COL], row[ADDRESS_COL]),
    axis=1
)

# Đếm số token trước/sau
df["n_tokens_before"] = df["tokens_before_stopwords"].apply(len)
df["n_tokens_after"] = df["processed_text_Final"].apply(len)
df["token_keep_ratio"] = df["n_tokens_after"] / df["n_tokens_before"].replace(0, pd.NA)

print("Số dòng hiện tại:", len(df))
print("Trung bình token trước lọc:", round(df["n_tokens_before"].mean(), 2))
print("Trung bình token sau lọc:", round(df["n_tokens_after"].mean(), 2))
print("Tỷ lệ token giữ lại trung bình:", round(df["token_keep_ratio"].mean(), 2))
print("Số dòng dưới 2 token sau preprocessing:", (df["n_tokens_after"] < 2).sum())

display(
    df[
        [
            COMMENT_COL,
            ADDRESS_COL,
            "processed_text_raw_phrase",
            "tokens_before_stopwords",
            "processed_text_Final",
            "n_tokens_before",
            "n_tokens_after",
            "token_keep_ratio"
        ]
    ].head(20)
)


Số dòng hiện tại: 64405
Trung bình token trước lọc: 56.8
Trung bình token sau lọc: 24.59
Tỷ lệ token giữ lại trung bình: 0.44
Số dòng dưới 2 token sau preprocessing: 662


,Comment_en,Address,processed_text_raw_phrase,tokens_before_stopwords,processed_text_Final,n_tokens_before,n_tokens_after,token_keep_ratio
0,It's about 10-15 minutes away from the old cit...,An Bang Beach,it about 10 15 minutes away from the old city ...,"[it, about, 10, 15, minutes, away, from, the, ...","[old, city, considered, one, best, beach, area...",135,52,0.385185
1,Quiet and quaint. Very laid back vive,An Bang Beach,quiet and quaint very laid back vive,"[quiet, and, quaint, very, laid, back, vive]","[quiet, quaint, laid, back]",7,4,0.571429
2,"It's a wonderful calm beach with clear water, ...",An Bang Beach,it a wonderful calm beach with clear_water so ...,"[it, a, wonderful, calm, beach, with, clear_wa...","[wonderful, calm, beach, clear_water, early, b...",32,13,0.406250
3,"While spending time in Hoi An, I was looking f...",An Bang Beach,while spending time in hoi an i was looking fo...,"[while, spending, time, in, hoi, an, i, was, l...","[spending, hoi, hoi, option, across, small, we...",108,50,0.462963
4,Hi everyone—October’s storm caused severe dama...,An Bang Beach,hi everyone october storm caused severe damage...,"[hi, everyone, october, storm, caused, severe,...","[everyone, october, storm, caused, severe, dam...",23,13,0.565217
5,"Clean, beautiful stretch of beach. Warm water,...",An Bang Beach,clean beautiful stretch of beach warm water so...,"[clean, beautiful, stretch, of, beach, warm, w...","[clean, beautiful, stretch, beach, warm, water...",74,35,0.472973
6,"A pristine, picturesque beach with warm water,...",An Bang Beach,a pristine picturesque beach with warm water p...,"[a, pristine, picturesque, beach, with, warm, ...","[pristine, picturesque, beach, warm, water, po...",32,19,0.593750
7,We ditched the crowds for An Bang and scored a...,An Bang Beach,we ditched the crowds for an bang and scored a...,"[we, ditched, the, crowds, for, an, bang, and,...","[ditched, crowd, bang, scored, quiet, stretch,...",37,21,0.567568
8,"An Bang Beach in Hoi An was, honestly, one of ...",An Bang Beach,an bang beach in hoi an was honestly one of my...,"[an, bang, beach, in, hoi, an, was, honestly, ...","[bang, beach, hoi, honestly, one, favorite, tr...",47,20,0.425532
9,An Bang Beach is a short Grab ride from Hoi An...,An Bang Beach,an bang beach is a short grab ride from hoi an...,"[an, bang, beach, is, a, short, grab, ride, fr...","[bang, beach, short, grab, ride, hoi, great, r...",103,36,0.349515


In [7]:
# --- 5. KIỂM TRA CỤM ĐÃ GHÉP ---
# Các token có dấu "_" là phrase đã được giữ lại.

phrase_tokens = []

for tokens in df["processed_text_Final"]:
    for token in tokens:
        if "_" in token:
            phrase_tokens.append(token)

phrase_counts = pd.Series(phrase_tokens).value_counts()

print("Số cụm phrase xuất hiện trong dữ liệu:", len(phrase_counts))
display(phrase_counts.head(80))


Số cụm phrase xuất hiện trong dữ liệu: 69


cable_car               1514
entrance_fee            1113
water_park               940
tour_guide               886
old_town                 845
                        ... 
visual_perspectivist       1
serious_i                  1
2016_12                    1
siu_hioi                   1
life_of_lukes              1
Name: count, Length: 69, dtype: int64

In [8]:
# --- 6. KIỂM TRA TOKEN PHỔ BIẾN SAU PREPROCESSING ---
# Nhìn bảng này để quyết định có cần thêm từ vào custom_stopwords hay không.

all_tokens = []

for tokens in df["processed_text_Final"]:
    all_tokens.extend(tokens)

token_counts = pd.Series(all_tokens).value_counts()

print("Số token duy nhất sau preprocessing:", len(token_counts))
display(token_counts.head(100))


Số token duy nhất sau preprocessing: 34520


visit         12643
people        10441
one           10238
beautiful     10199
tourist        9543
              ...  
definitely     2398
morning        2379
everything     2375
big            2371
chi            2338
Name: count, Length: 100, dtype: int64

In [9]:
# --- 7. LỌC DÒNG QUÁ ÍT TOKEN + TẠO CỘT processed_text ĐỂ DỄ XEM ---
# Chỉ xóa những dòng còn dưới 2 token sau preprocessing.

before_filter = len(df)

df = df[df["processed_text_Final"].apply(len) >= 2].copy()

after_filter = len(df)

print("Số dòng trước khi lọc:", before_filter)
print("Số dòng sau khi lọc:", after_filter)
print("Số dòng bị xóa:", before_filter - after_filter)

# processed_text: dạng chuỗi để nhìn nhanh
# processed_text_Final: dạng list token để đưa vào LDA
df["processed_text"] = df["processed_text_Final"].apply(lambda tokens: " ".join(tokens))

keep_cols = ["Username", "Rating", "Address", "Comment_en", "processed_text", "processed_text_Final"]
keep_cols = [col for col in keep_cols if col in df.columns]
df = df[keep_cols].copy()

display(df[["Comment_en", "processed_text", "processed_text_Final"]].head())


Số dòng trước khi lọc: 64405
Số dòng sau khi lọc: 63743
Số dòng bị xóa: 662


,Comment_en,processed_text,processed_text_Final
0,It's about 10-15 minutes away from the old cit...,old city considered one best beach area beach ...,"[old, city, considered, one, best, beach, area..."
1,Quiet and quaint. Very laid back vive,quiet quaint laid back,"[quiet, quaint, laid, back]"
2,"It's a wonderful calm beach with clear water, ...",wonderful calm beach clear_water early beach r...,"[wonderful, calm, beach, clear_water, early, b..."
3,"While spending time in Hoi An, I was looking f...",spending hoi hoi option across small well orga...,"[spending, hoi, hoi, option, across, small, we..."
4,Hi everyone—October’s storm caused severe dama...,everyone october storm caused severe damage oc...,"[everyone, october, storm, caused, severe, dam..."


LDA

In [10]:
# --- 3. TẠO DICTIONARY VÀ CORPUS ---
dictionary = corpora.Dictionary(df['processed_text_Final'])
# Lọc bỏ các từ quá hiếm hoặc quá phổ biến (tùy chọn để tăng chất lượng)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(text) for text in df['processed_text_Final']]
print(corpus[:2])  # In hai dòng đầu tiên

[[(0, 2), (1, 1), (2, 1), (3, 2), (4, 1), (5, 1), (6, 1), (7, 3), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 2), (17, 1), (18, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1), (34, 2), (35, 1), (36, 1), (37, 1), (38, 1), (39, 1), (40, 1), (41, 1), (42, 1), (43, 1), (44, 1), (45, 1)], [(46, 1), (47, 1), (48, 1), (49, 1)]]


In [11]:
# Kiểm tra số lượng từ khóa và tài liệu
print(f"Số lượng từ khóa duy nhất: {len(dictionary)}")
print(f"Số lượng tài liệu: {len(corpus)}")

# In thử một số từ trong dictionary
print("\nMột số từ khóa trong dictionary:")

for token_id, token in list(dictionary.items())[:50]:
    print(f"{token_id}: {token}")

Số lượng từ khóa duy nhất: 10756
Số lượng tài liệu: 63743

Một số từ khóa trong dictionary:
0: along
1: area
2: bathe
3: beach
4: beautiful
5: bed
6: best
7: cafe
8: calm
9: city
10: coast
11: coastline
12: coffee
13: considered
14: cozy
15: crowded
16: drink
17: eat
18: feeling
19: food
20: free
21: great
22: long
23: ocean
24: often
25: old
26: one
27: order
28: particularly
29: relaxing
30: resort
31: rest
32: restaurant
33: sand
34: sea
35: sit
36: soft
37: sunset
38: tonight
39: umbrella
40: use
41: view
42: walk
43: water
44: way
45: wide
46: back
47: laid
48: quaint
49: quiet


In [12]:
# Kiểm tra coherence score cho từng số lượng chủ đề
coherence_scores = []
for num_topics in range(5, 15):  # Thử từ 5 đến 20 chủ đề
    lda_model = LdaModel(corpus=corpus,
                         id2word=dictionary,
                         num_topics=num_topics,
                         random_state=42,
                         passes=20,
                         alpha='asymmetric',
                         eta='auto')
    
    coherence_model = CoherenceModel(model=lda_model, texts=df['processed_text_Final'], dictionary=dictionary, coherence='c_v')
    score = coherence_model.get_coherence()
    coherence_scores.append((num_topics, score))

# In kết quả
for num, score in coherence_scores:
    print(f"Chủ đề: {num} - Coherence Score: {score:.4f}")

Chủ đề: 5 - Coherence Score: 0.5412
Chủ đề: 6 - Coherence Score: 0.5163
Chủ đề: 7 - Coherence Score: 0.4889
Chủ đề: 8 - Coherence Score: 0.4717
Chủ đề: 9 - Coherence Score: 0.5056
Chủ đề: 10 - Coherence Score: 0.4981
Chủ đề: 11 - Coherence Score: 0.4751
Chủ đề: 12 - Coherence Score: 0.5027
Chủ đề: 13 - Coherence Score: 0.5078
Chủ đề: 14 - Coherence Score: 0.5017


Topics LDA

In [13]:
#LDA5: 0.5225
lda_5_model = LdaModel(corpus=corpus,
                         id2word=dictionary,
                         num_topics=5,
                         random_state=42,
                         passes=20,
                         alpha='asymmetric',
                         eta='auto')

# Trực quan hóa kết quả LDA
pyLDAvis.enable_notebook()
lda_vis_data_2 = gensimvis.prepare(lda_5_model , corpus, dictionary,mds='mmds', lambda_step=0.1)
pyLDAvis.display(lda_vis_data_2)

In [14]:
# In ra các từ phổ biến cho từng chủ đề
print("\n--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---")
for idx, topic in lda_5_model.print_topics(-1):
    print(f"Chủ đề {idx}: {topic}")


--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---
Chủ đề 0: 0.015*"beautiful" + 0.013*"nice" + 0.010*"tour" + 0.010*"good" + 0.010*"view" + 0.009*"experience" + 0.009*"water" + 0.009*"boat" + 0.009*"great" + 0.008*"cave"
Chủ đề 1: 0.020*"ticket" + 0.013*"000" + 0.010*"price" + 0.010*"tourist" + 0.009*"buy" + 0.009*"pay" + 0.009*"money" + 0.009*"people" + 0.009*"one" + 0.008*"jeep"
Chủ đề 2: 0.019*"ride" + 0.019*"dune" + 0.017*"park" + 0.012*"show" + 0.010*"kid" + 0.009*"game" + 0.009*"water_park" + 0.008*"one" + 0.008*"like" + 0.007*"office"
Chủ đề 3: 0.030*"war" + 0.022*"museum" + 0.020*"vietnam" + 0.019*"history" + 0.015*"visit" + 0.014*"interesting" + 0.012*"vietnamese" + 0.008*"american" + 0.008*"well" + 0.007*"one"
Chủ đề 4: 0.025*"temple" + 0.019*"beautiful" + 0.019*"visit" + 0.019*"city" + 0.015*"architecture" + 0.015*"building" + 0.012*"bridge" + 0.011*"tower" + 0.010*"worth" + 0.010*"tourist"


In [15]:
#LDA6: 0.5044
lda_6_model = LdaModel(corpus=corpus,
                         id2word=dictionary,
                         num_topics=6,
                         random_state=42,
                         passes=20,
                         alpha='asymmetric',
                         eta='auto')

# Trực quan hóa kết quả LDA
pyLDAvis.enable_notebook()
lda_vis_data_2 = gensimvis.prepare(lda_6_model , corpus, dictionary,mds='mmds', lambda_step=0.1)
pyLDAvis.display(lda_vis_data_2)

In [16]:
# In ra các từ phổ biến cho từng chủ đề
print("\n--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---")
for idx, topic in lda_6_model.print_topics(-1):
    print(f"Chủ đề {idx}: {topic}")


--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---
Chủ đề 0: 0.015*"ride" + 0.014*"beautiful" + 0.013*"nice" + 0.011*"water" + 0.010*"good" + 0.009*"fun" + 0.009*"great" + 0.009*"view" + 0.009*"boat" + 0.008*"experience"
Chủ đề 1: 0.021*"ticket" + 0.014*"000" + 0.011*"price" + 0.011*"money" + 0.010*"buy" + 0.010*"pay" + 0.010*"tourist" + 0.009*"one" + 0.008*"people" + 0.008*"back"
Chủ đề 2: 0.024*"dune" + 0.012*"park" + 0.011*"show" + 0.011*"water_park" + 0.010*"like" + 0.009*"one" + 0.009*"people" + 0.009*"office" + 0.009*"line" + 0.008*"post"
Chủ đề 3: 0.030*"war" + 0.022*"museum" + 0.022*"visit" + 0.019*"vietnam" + 0.019*"history" + 0.016*"interesting" + 0.011*"jeep" + 0.011*"tour" + 0.011*"worth" + 0.010*"well"
Chủ đề 4: 0.022*"beautiful" + 0.022*"temple" + 0.019*"visit" + 0.018*"city" + 0.017*"architecture" + 0.016*"building" + 0.013*"bridge" + 0.012*"tourist" + 0.010*"old" + 0.010*"photo"
Chủ đề 5: 0.033*"trang" + 0.024*"tower" + 0.023*"nha" + 0.016*"city" + 0.015*"vietnam" + 0.015*"south" + 0

In [17]:
#LDA7: 0.5078
lda_7_model = LdaModel(corpus=corpus,
                         id2word=dictionary,
                         num_topics=7,
                         random_state=42,
                         passes=20,
                         alpha='asymmetric',
                         eta='auto')

# Trực quan hóa kết quả LDA
pyLDAvis.enable_notebook()
lda_vis_data_2 = gensimvis.prepare(lda_7_model , corpus, dictionary,mds='mmds', lambda_step=0.1)
pyLDAvis.display(lda_vis_data_2)

In [18]:
# In ra các từ phổ biến cho từng chủ đề
print("\n--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---")
for idx, topic in lda_7_model.print_topics(-1):
    print(f"Chủ đề {idx}: {topic}")


--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---
Chủ đề 0: 0.018*"beautiful" + 0.011*"experience" + 0.010*"people" + 0.010*"jeep" + 0.008*"like" + 0.007*"tourist" + 0.007*"local" + 0.007*"city" + 0.007*"great" + 0.007*"vietnam"
Chủ đề 1: 0.020*"ticket" + 0.013*"000" + 0.011*"people" + 0.010*"price" + 0.010*"money" + 0.009*"one" + 0.009*"buy" + 0.009*"pay" + 0.009*"like" + 0.009*"tourist"
Chủ đề 2: 0.022*"ride" + 0.016*"dune" + 0.014*"park" + 0.012*"nice" + 0.010*"water" + 0.009*"one" + 0.009*"tour" + 0.009*"cave" + 0.009*"good" + 0.008*"view"
Chủ đề 3: 0.029*"war" + 0.022*"visit" + 0.022*"museum" + 0.020*"history" + 0.016*"interesting" + 0.015*"vietnam" + 0.011*"tour" + 0.010*"well" + 0.010*"room" + 0.010*"worth"
Chủ đề 4: 0.026*"building" + 0.021*"visit" + 0.020*"beautiful" + 0.019*"architecture" + 0.019*"temple" + 0.018*"post" + 0.016*"city" + 0.015*"photo" + 0.015*"worth" + 0.015*"bridge"
Chủ đề 5: 0.029*"closed" + 0.027*"year" + 0.022*"line" + 0.022*"open" + 0.015*"allowed" + 0.014*"long" + 0.

In [19]:
#LDA8: 0.5211
lda_8_model = LdaModel(corpus=corpus,
                         id2word=dictionary,
                         num_topics=8,
                         random_state=42,
                         passes=20,
                         alpha='asymmetric',
                         eta='auto')

# Trực quan hóa kết quả LDA
pyLDAvis.enable_notebook()
lda_vis_data_2 = gensimvis.prepare(lda_8_model , corpus, dictionary,mds='mmds', lambda_step=0.1)
pyLDAvis.display(lda_vis_data_2)

In [20]:
# In ra các từ phổ biến cho từng chủ đề
print("\n--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---")
for idx, topic in lda_8_model.print_topics(-1):
    print(f"Chủ đề {idx}: {topic}")


--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---
Chủ đề 0: 0.022*"beautiful" + 0.014*"nice" + 0.012*"view" + 0.011*"cave" + 0.011*"experience" + 0.010*"great" + 0.009*"walk" + 0.009*"good" + 0.009*"sand" + 0.009*"people"
Chủ đề 1: 0.024*"ticket" + 0.016*"000" + 0.012*"price" + 0.011*"buy" + 0.011*"pay" + 0.011*"money" + 0.010*"back" + 0.009*"one" + 0.008*"vnd" + 0.008*"people"
Chủ đề 2: 0.024*"park" + 0.019*"water" + 0.018*"ride" + 0.017*"show" + 0.014*"water_park" + 0.012*"office" + 0.012*"beach" + 0.011*"trang" + 0.009*"one" + 0.009*"kid"
Chủ đề 3: 0.022*"war" + 0.019*"dune" + 0.016*"visit" + 0.016*"tour" + 0.014*"museum" + 0.013*"would" + 0.011*"interesting" + 0.011*"worth" + 0.010*"one" + 0.009*"ride"
Chủ đề 4: 0.039*"building" + 0.026*"visit" + 0.022*"beautiful" + 0.019*"worth" + 0.018*"temple" + 0.017*"photo" + 0.017*"post" + 0.015*"bridge" + 0.015*"architecture" + 0.013*"old"
Chủ đề 5: 0.073*"nothing" + 0.031*"special" + 0.030*"story" + 0.025*"year" + 0.024*"like" + 0.019*"tam" + 0.015*"tou

In [21]:
#LDA9: 0.5043
lda_9_model = LdaModel(corpus=corpus,
                         id2word=dictionary,
                         num_topics=9,
                         random_state=42,
                         passes=20,
                         alpha='asymmetric',
                         eta='auto')

# Trực quan hóa kết quả LDA
pyLDAvis.enable_notebook()
lda_vis_data_2 = gensimvis.prepare(lda_9_model , corpus, dictionary,mds='mmds', lambda_step=0.1)
pyLDAvis.display(lda_vis_data_2)

In [22]:
# In ra các từ phổ biến cho từng chủ đề
print("\n--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---")
for idx, topic in lda_9_model.print_topics(-1):
    print(f"Chủ đề {idx}: {topic}")


--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---
Chủ đề 0: 0.018*"beautiful" + 0.014*"nice" + 0.014*"water" + 0.010*"cave" + 0.010*"fun" + 0.010*"good" + 0.010*"view" + 0.009*"experience" + 0.009*"great" + 0.009*"people"
Chủ đề 1: 0.020*"ticket" + 0.020*"000" + 0.015*"price" + 0.014*"buy" + 0.013*"tourist" + 0.011*"pay" + 0.011*"vnd" + 0.009*"dong" + 0.009*"good" + 0.009*"food"
Chủ đề 2: 0.039*"dune" + 0.029*"tour" + 0.020*"quad" + 0.020*"show" + 0.015*"office" + 0.014*"post" + 0.013*"hotel" + 0.011*"bus" + 0.009*"trip" + 0.009*"back"
Chủ đề 3: 0.028*"war" + 0.023*"visit" + 0.017*"museum" + 0.017*"history" + 0.017*"interesting" + 0.014*"worth" + 0.011*"well" + 0.010*"room" + 0.010*"vietnam" + 0.009*"would"
Chủ đề 4: 0.029*"temple" + 0.028*"beautiful" + 0.027*"jeep" + 0.023*"visit" + 0.016*"bridge" + 0.015*"city" + 0.015*"photo" + 0.015*"architecture" + 0.013*"worth" + 0.013*"tourist"
Chủ đề 5: 0.073*"year" + 0.061*"old" + 0.028*"new" + 0.026*"display" + 0.015*"ago" + 0.014*"like" + 0.014*"tea" + 0

In [23]:
#LDA10: 0.5033
lda_10_model = LdaModel(corpus=corpus,
                         id2word=dictionary,
                         num_topics=10,
                         random_state=42,
                         passes=20,
                         alpha='asymmetric',
                         eta='auto')

# Trực quan hóa kết quả LDA
pyLDAvis.enable_notebook()
lda_vis_data_2 = gensimvis.prepare(lda_10_model , corpus, dictionary,mds='mmds', lambda_step=0.1)
pyLDAvis.display(lda_vis_data_2)

In [24]:
# In ra các từ phổ biến cho từng chủ đề
print("\n--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---")
for idx, topic in lda_10_model.print_topics(-1):
    print(f"Chủ đề {idx}: {topic}")


--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---
Chủ đề 0: 0.017*"beautiful" + 0.016*"like" + 0.013*"people" + 0.012*"experience" + 0.010*"slide" + 0.010*"sand_dunes" + 0.010*"feel" + 0.009*"lake" + 0.008*"tourist" + 0.008*"elevator"
Chủ đề 1: 0.018*"good" + 0.018*"food" + 0.017*"tourist" + 0.014*"price" + 0.014*"people" + 0.013*"shop" + 0.011*"buy" + 0.011*"like" + 0.011*"aquarium" + 0.010*"nice"
Chủ đề 2: 0.021*"dune" + 0.021*"ride" + 0.016*"park" + 0.014*"one" + 0.012*"would" + 0.009*"water_park" + 0.009*"even" + 0.009*"people" + 0.009*"like" + 0.009*"staff"
Chủ đề 3: 0.045*"war" + 0.033*"museum" + 0.025*"history" + 0.024*"vietnam" + 0.021*"visit" + 0.021*"tour" + 0.021*"interesting" + 0.017*"jeep" + 0.015*"guide" + 0.014*"vietnamese"
Chủ đề 4: 0.035*"visit" + 0.034*"worth" + 0.028*"nice" + 0.025*"view" + 0.025*"beautiful" + 0.021*"temple" + 0.019*"photo" + 0.012*"walk" + 0.012*"bridge" + 0.012*"tourist"
Chủ đề 5: 0.045*"show" + 0.039*"hot" + 0.038*"game" + 0.037*"water" + 0.028*"fun" + 0.023*"

In [25]:
#LDA11: 0.5215
lda_11_model = LdaModel(corpus=corpus,
                         id2word=dictionary,
                         num_topics=11,
                         random_state=42,
                         passes=20,
                         alpha='asymmetric',
                         eta='auto')

# Trực quan hóa kết quả LDA
pyLDAvis.enable_notebook()
lda_vis_data_2 = gensimvis.prepare(lda_11_model , corpus, dictionary,mds='mmds', lambda_step=0.1)
pyLDAvis.display(lda_vis_data_2)

In [26]:
# In ra các từ phổ biến cho từng chủ đề
print("\n--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---")
for idx, topic in lda_11_model.print_topics(-1):
    print(f"Chủ đề {idx}: {topic}")


--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---
Chủ đề 0: 0.020*"cave" + 0.019*"beautiful" + 0.013*"experience" + 0.012*"people" + 0.011*"like" + 0.010*"slide" + 0.010*"mountain" + 0.009*"lake" + 0.009*"feel" + 0.008*"hanoi"
Chủ đề 1: 0.031*"ticket" + 0.021*"000" + 0.016*"price" + 0.014*"buy" + 0.013*"pay" + 0.012*"tourist" + 0.011*"vnd" + 0.010*"atv" + 0.010*"dong" + 0.010*"walk"
Chủ đề 2: 0.036*"ride" + 0.033*"park" + 0.030*"dune" + 0.018*"show" + 0.017*"fun" + 0.015*"quad" + 0.015*"kid" + 0.015*"good" + 0.014*"water" + 0.013*"game"
Chủ đề 3: 0.038*"war" + 0.026*"museum" + 0.022*"tour" + 0.022*"history" + 0.020*"visit" + 0.020*"interesting" + 0.018*"vietnam" + 0.015*"jeep" + 0.013*"guide" + 0.012*"well"
Chủ đề 4: 0.035*"visit" + 0.034*"beautiful" + 0.033*"worth" + 0.031*"view" + 0.028*"nice" + 0.022*"temple" + 0.021*"photo" + 0.015*"bridge" + 0.013*"tourist" + 0.012*"great"
Chủ đề 5: 0.055*"hot" + 0.053*"story" + 0.030*"boring" + 0.026*"season" + 0.025*"boat_ride" + 0.023*"weather" + 0.018*"tir

In [27]:
#LDA12: 0.5065
lda_12_model = LdaModel(corpus=corpus,
                         id2word=dictionary,
                         num_topics=12,
                         random_state=42,
                         passes=20,
                         alpha='asymmetric',
                         eta='auto')

# Trực quan hóa kết quả LDA
pyLDAvis.enable_notebook()
lda_vis_data_2 = gensimvis.prepare(lda_12_model , corpus, dictionary,mds='mmds', lambda_step=0.1)
pyLDAvis.display(lda_vis_data_2)

In [28]:
# In ra các từ phổ biến cho từng chủ đề
print("\n--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---")
for idx, topic in lda_12_model.print_topics(-1):
    print(f"Chủ đề {idx}: {topic}")


--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---
Chủ đề 0: 0.021*"beautiful" + 0.015*"office" + 0.014*"post" + 0.012*"like" + 0.011*"experience" + 0.010*"tourist" + 0.010*"people" + 0.010*"lake" + 0.008*"hanoi" + 0.008*"feel"
Chủ đề 1: 0.018*"good" + 0.018*"food" + 0.018*"tourist" + 0.016*"price" + 0.012*"people" + 0.012*"aquarium" + 0.012*"like" + 0.011*"buy" + 0.011*"shop" + 0.010*"nice"
Chủ đề 2: 0.029*"ride" + 0.025*"dune" + 0.024*"park" + 0.016*"one" + 0.014*"show" + 0.013*"would" + 0.011*"water_park" + 0.010*"people" + 0.009*"like" + 0.008*"line"
Chủ đề 3: 0.048*"war" + 0.036*"museum" + 0.029*"history" + 0.025*"vietnam" + 0.023*"interesting" + 0.021*"visit" + 0.016*"tour" + 0.015*"guide" + 0.015*"vietnamese" + 0.013*"well"
Chủ đề 4: 0.038*"nice" + 0.032*"worth" + 0.031*"visit" + 0.026*"view" + 0.023*"beautiful" + 0.021*"temple" + 0.018*"photo" + 0.015*"walk" + 0.014*"good" + 0.014*"slide"
Chủ đề 5: 0.041*"hot" + 0.040*"game" + 0.027*"child" + 0.021*"kid" + 0.020*"play" + 0.019*"story" + 0.01

In [29]:
#LDA13: 0.5072
lda_13_model = LdaModel(corpus=corpus,
                         id2word=dictionary,
                         num_topics=13,
                         random_state=42,
                         passes=20,
                         alpha='asymmetric',
                         eta='auto')

# Trực quan hóa kết quả LDA
pyLDAvis.enable_notebook()
lda_vis_data_2 = gensimvis.prepare(lda_13_model , corpus, dictionary,mds='mmds', lambda_step=0.1)
pyLDAvis.display(lda_vis_data_2)

In [30]:
# In ra các từ phổ biến cho từng chủ đề
print("\n--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---")
for idx, topic in lda_13_model.print_topics(-1):
    print(f"Chủ đề {idx}: {topic}")


--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---
Chủ đề 0: 0.047*"war" + 0.022*"cave" + 0.016*"people" + 0.015*"beautiful" + 0.012*"like" + 0.011*"slide" + 0.010*"vietnam" + 0.009*"experience" + 0.008*"mountain" + 0.008*"feel"
Chủ đề 1: 0.033*"ticket" + 0.022*"000" + 0.016*"price" + 0.015*"buy" + 0.015*"pay" + 0.014*"jeep" + 0.013*"money" + 0.011*"vnd" + 0.011*"one" + 0.010*"dong"
Chủ đề 2: 0.057*"dune" + 0.038*"show" + 0.023*"staff" + 0.021*"office" + 0.020*"water" + 0.016*"service" + 0.013*"park" + 0.013*"good" + 0.012*"bad" + 0.011*"room"
Chủ đề 3: 0.036*"ride" + 0.023*"tour" + 0.021*"park" + 0.014*"good" + 0.013*"fun" + 0.012*"would" + 0.011*"kid" + 0.010*"water_park" + 0.010*"visit" + 0.010*"great"
Chủ đề 4: 0.054*"beautiful" + 0.047*"temple" + 0.040*"visit" + 0.029*"view" + 0.022*"worth" + 0.021*"bridge" + 0.021*"must" + 0.020*"photo" + 0.016*"architecture" + 0.013*"great"
Chủ đề 5: 0.042*"food" + 0.038*"good" + 0.034*"shop" + 0.029*"street" + 0.027*"restaurant" + 0.018*"nice" + 0.017*"souve

In [31]:
#LDA14: 0.5139
lda_14_model = LdaModel(corpus=corpus,
                         id2word=dictionary,
                         num_topics=14,
                         random_state=42,
                         passes=20,
                         alpha='asymmetric',
                         eta='auto')

# Trực quan hóa kết quả LDA
pyLDAvis.enable_notebook()
lda_vis_data_2 = gensimvis.prepare(lda_14_model , corpus, dictionary,mds='mmds', lambda_step=0.1)
pyLDAvis.display(lda_vis_data_2)

In [32]:
# In ra các từ phổ biến cho từng chủ đề
print("\n--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---")
for idx, topic in lda_14_model.print_topics(-1):
    print(f"Chủ đề {idx}: {topic}")


--- CÁC TỪ PHỔ BIẾN THEO CHỦ ĐỀ ---
Chủ đề 0: 0.027*"beautiful" + 0.021*"cave" + 0.014*"experience" + 0.012*"sunrise" + 0.011*"water" + 0.011*"sand_dunes" + 0.010*"river" + 0.010*"lake" + 0.009*"mountain" + 0.009*"people"
Chủ đề 1: 0.028*"000" + 0.025*"ticket" + 0.021*"price" + 0.019*"buy" + 0.016*"food" + 0.015*"vnd" + 0.014*"good" + 0.014*"shop" + 0.013*"dong" + 0.012*"aquarium"
Chủ đề 2: 0.059*"dune" + 0.022*"office" + 0.021*"post" + 0.020*"room" + 0.016*"show" + 0.015*"service" + 0.014*"bad" + 0.012*"still" + 0.011*"floor" + 0.010*"year"
Chủ đề 3: 0.038*"ride" + 0.029*"park" + 0.015*"kid" + 0.015*"fun" + 0.014*"child" + 0.014*"game" + 0.014*"water_park" + 0.013*"good" + 0.012*"show" + 0.011*"line"
Chủ đề 4: 0.058*"temple" + 0.052*"nice" + 0.046*"beautiful" + 0.040*"view" + 0.035*"photo" + 0.023*"bridge" + 0.022*"visit" + 0.020*"walk" + 0.018*"tourist" + 0.016*"worth"
Chủ đề 5: 0.054*"like" + 0.046*"nothing" + 0.024*"tourist" + 0.023*"would" + 0.022*"people" + 0.021*"little" + 0.01